In [8]:
#输出dict文件的最长的一个单词的长度，并以此作为”定长数组储存词典” 方法的每个单词长度
#统计文档中的单词总数
#统计文档中的字符总数
def find_longest_word_length(file_path):
    with open(file_path, 'r', encoding='utf-8') as file:
        words = file.read().strip().split()  
        longest_word = max(words, key=len)  
        return len(longest_word) 

def count_total_characters(file_path):
    with open(file_path, 'r', encoding='utf-8') as file:
        content = file.read().replace(" ", "")  # 去除空格后统计字符数
        return len(content)

def count_total_words(file_path):
    with open(file_path, 'r', encoding='utf-8') as file:
        words = file.read().strip().split()
        return len(words)

if __name__ == "__main__":
    file_path = r"D:\code\natural_language_processing\lab5\dict.txt"  
    longest_length = find_longest_word_length(file_path)
    total_characters = count_total_characters(file_path)
    total_words = count_total_words(file_path)

    print(f"最长单词的字符长度是: {longest_length}")
    print(f"文档总字符数是: {total_characters}")
    print(f"文档中共有单词数: {total_words}")

最长单词的字符长度是: 22
文档总字符数是: 654452
文档中共有单词数: 87980


由上以上代码的输出结果，假设用储存每个单词的文档频率和指向倒排记录表的指针的空间大小都是4B
每个单词需要22B的空间来储存
可以计算出采用顶常熟组储存的空间开销为：
（22+4+4）×87980 = 2639400B

In [18]:
#将词典压缩成单一字符串
def compress_dictionary(file_path):
    with open(file_path, 'r', encoding='utf-8') as file:
        content = file.read().strip()
    
    words = content.split()
    compressed = ''.join(f"{len(word)}{word}" for word in words)
    
    return compressed


if __name__ == "__main__":
    file_path = r"D:\code\natural_language_processing\lab5\dict.txt"
    result = compress_dictionary(file_path)  
    output_path = r"D:\code\natural_language_processing\lab5\compressed_dict.txt"
    with open(output_path, 'w', encoding='utf-8') as f:
        f.write(result)

In [ ]:
import time
import re
import os

def is_valid_lowercase(s):
    return bool(re.fullmatch(r'[a-z]+', s))

def longest_common_prefix(words):
    if not words:
        return ''
    min_word = min(words)
    max_word = max(words)
    i = 0
    while i < len(min_word) and min_word[i] == max_word[i]:
        i += 1
    return min_word[:i]

def group_by_prefix(words):
    groups = []
    i = 0
    n = len(words)

    while i < n:
        j = i + 1
        prefix = words[i]
        # 查找最大可能的公共前缀组
        while j < n:
            current_prefix = longest_common_prefix([prefix, words[j]])
            if not current_prefix or len(current_prefix) < 1:  # 设置最小长度如2
                break
            prefix = current_prefix
            j += 1
        groups.append((prefix, words[i:j]))
        i = j
    return groups

def compress_block(block_words):
    block_words = sorted(block_words)  
    compressed = ""
    prefixes_in_block = []

    groups = group_by_prefix(block_words)  

    for prefix, group in groups:
        if len(group) > 1:
            # 多个词共享前缀
            compressed += f"{len(prefix)}{prefix}*"
            prefixes_in_block.append(prefix)
            for word in group:
                suffix = word[len(prefix):]
                compressed += f"{len(suffix)}${suffix}"
        else:
            # 只有一个词，自己作为前缀
            word = group[0]
            compressed += f"{len(word)}{word}*"
            prefixes_in_block.append(word)

    return compressed, prefixes_in_block

def build_index(compressed_str, k):
    index_positions = []
    result_blocks = []
    prefixes_per_block = []

    pos = 0
    words_with_length = re.findall(r'(\d+)([a-z]+)', compressed_str)
    words = [w[1] for w in words_with_length]

    for i in range(0, len(words), k):
        block_words = words[i:i + k]
        start_pos = pos
        index_positions.append(start_pos)

        # 块压缩并进行分组处理
        block_str, prefixes = compress_block(block_words)
        result_blocks.append(block_str)
        prefixes_per_block.append(prefixes)

        pos += len(block_str)

    full_compressed = ''.join(result_blocks)
    return full_compressed, index_positions, len(index_positions)

def binary_search_query(compressed_str, index_positions, query_word):
    left, right = 0, len(index_positions) - 1
    found = False

    while left <= right:
        mid = (left + right) // 2
        start = index_positions[mid]
        end = index_positions[mid + 1] if mid + 1 < len(index_positions) else len(compressed_str)
        block_str = compressed_str[start:end]

        entries = re.findall(r'(\d+)(\$|\*)?([a-z]*)', block_str)

        current_prefix = None
        for length, flag, word in entries:
            if flag == '*':
                current_prefix = word
            elif flag == '$':
                if current_prefix is None:
                    continue  
                suffix_len = int(length)
                suffix = word[:suffix_len]
                candidate = current_prefix + suffix
                if query_word == candidate:
                    found = True
                    break
            elif word:
                candidate = word
                if query_word == candidate:
                    found = True
                    break
        if found:
            break
        elif mid + 1 < len(index_positions) and query_word > block_str:
            left = mid + 1
        else:
            right = mid - 1

    return found

def main():
    try:
        k = int(input("请输入块大小 k（1~87980）: "))
        if not (1 <= k <= 87980):
            print("错误：k 必须在 1 到 87980 之间")
            return

        query_word = input("请输入要查询的单词（仅限小写字母）: ").strip()
        if not is_valid_lowercase(query_word):
            print("错误：查询词必须为小写字母组成")
            return

        input_path = r"D:\code\natural_language_processing\lab5\compressed_dict.txt"
        output_path = fr"D:\code\natural_language_processing\lab5\compressed_dict_k={k}.txt"

        with open(input_path, 'r', encoding='utf-8') as f:
            compressed_str = f.read().strip()

        full_compressed, index_positions, blocknumber = build_index(compressed_str, k)

        with open(output_path, 'w', encoding='utf-8') as f:
            f.write(full_compressed)

        print(f"压缩后文件已保存至：{output_path}")

        iterations = 1000
        start_time = time.perf_counter()  
        for _ in range(iterations):
            binary_search_query(full_compressed, index_positions, query_word)
        end_time = time.perf_counter()

        avg_time_ms = ((end_time - start_time) * 1000) / iterations  
        print(f"平均查询时间为：{avg_time_ms:.4f} 毫秒")  

        new_length = len(full_compressed)
        overhead = (87980 + blocknumber) * 4
        original_size = 2639400  

        compression_rate = (new_length + overhead) / original_size * 100
        print(f"压缩率为：{compression_rate:.2f}%")

    except Exception as e:
        print(f"发生错误：{e}")

if __name__ == "__main__":
    main()

压缩后文件已保存至：D:\code\natural_language_processing\lab5\compressed_dict_k=10000.txt
平均查询时间为：8.0882 毫秒
压缩率为：41.81%


In [ ]:
import time  # 导入时间模块，用于计时操作
import re    # 导入正则表达式模块，用于字符串匹配和解析
import os    # 导入操作系统模块，用于文件路径等操作

def is_valid_lowercase(s):  # 检查输入字符串是否为全小写字母组成
    return bool(re.fullmatch(r'[a-z]+', s))  # 使用正则判断是否符合要求

def longest_common_prefix(words):  # 获取一组词的最长公共前缀
    if not words:  # 如果列表为空
        return ''  # 返回空字符串
    min_word = min(words)  # 找出字典序最小的单词
    max_word = max(words)  # 找出字典序最大的单词
    i = 0  # 初始化索引
    while i < len(min_word) and min_word[i] == max_word[i]:  # 循环直到字符不一致或到末尾
        i += 1  # 索引递增
    return min_word[:i]  # 返回公共前缀

def group_by_prefix(words):  # 根据最长公共前缀对词进行分组
    groups = []  # 存储结果分组
    i = 0  # 当前起始位置
    n = len(words)  # 总词数

    while i < n:  # 遍历所有词
        j = i + 1  # 下一个词的位置
        prefix = words[i]  # 当前组的初始前缀
        # 查找最大可能的公共前缀组
        while j < n:
            current_prefix = longest_common_prefix([prefix, words[j]])  # 计算当前与下一个词的前缀
            if not current_prefix or len(current_prefix) < 1:  # 如果没有公共前缀或长度小于设定值（如2）
                break  # 停止扩展该组
            prefix = current_prefix  # 更新当前前缀
            j += 1  # 移动到下一个词
        groups.append((prefix, words[i:j]))  # 将当前组加入结果
        i = j  # 更新起始位置
    return groups  # 返回分组结果

def compress_block(block_words):  # 对一个块内的词进行压缩处理
    block_words = sorted(block_words)   # 排序以便于分组
    compressed = ""  # 存储压缩后的字符串
    prefixes_in_block = []  # 存储每个前缀

    groups = group_by_prefix(block_words)  # 分组

    for prefix, group in groups:  # 遍历每个组
        if len(group) > 1:  # 如果有多个词共享前缀
            # 多个词共享前缀
            compressed += f"{len(prefix)}{prefix}*"
            prefixes_in_block.append(prefix)
            for word in group:
                suffix = word[len(prefix):]
                compressed += f"{len(suffix)}${suffix}"
        else:
            # 只有一个词，自己作为前缀
            word = group[0]
            compressed += f"{len(word)}{word}*"
            prefixes_in_block.append(word)

    return compressed, prefixes_in_block  # 返回压缩字符串和前缀列表

def build_index(compressed_str, k):  # 构建索引并分块压缩
    index_positions = []  # 存储每一块在压缩字符串中的起始位置
    result_blocks = []  # 存储压缩后各块的结果
    prefixes_per_block = []  # 存储每个块的前缀

    pos = 0  # 当前位置指针
    words_with_length = re.findall(r'(\d+)([a-z]+)', compressed_str)  # 提取带长度的词
    words = [w[1] for w in words_with_length]  # 提取实际单词

    for i in range(0, len(words), k):  # 每k个词作为一个块
        block_words = words[i:i + k]  # 当前块的词
        start_pos = pos  # 当前块起始位置
        index_positions.append(start_pos)  # 添加到索引中

        # 块压缩并进行分组处理
        block_str, prefixes = compress_block(block_words)  # 压缩当前块
        result_blocks.append(block_str)  # 添加到结果中
        prefixes_per_block.append(prefixes)  # 添加当前块的前缀

        pos += len(block_str)  # 更新当前位置

    full_compressed = ''.join(result_blocks)  # 合并所有块的压缩结果
    return full_compressed, index_positions, len(index_positions)  # 返回完整压缩串、索引和块数量

def binary_search_query(compressed_str, index_positions, query_word):  # 在压缩字符串中进行二分查找
    left, right = 0, len(index_positions) - 1  # 设置左右边界
    found = False  # 是否找到标志

    while left <= right:
        mid = (left + right) // 2  # 中间位置
        start = index_positions[mid]  # 当前块起始位置
        end = index_positions[mid + 1] if mid + 1 < len(index_positions) else len(compressed_str)  # 结束位置
        block_str = compressed_str[start:end]  # 提取当前块的字符串

        entries = re.findall(r'(\d+)(\$|\*)?([a-z]*)', block_str)  # 解析块内容

        current_prefix = None
        for length, flag, word in entries:
            if flag == '*':  # 表示前缀
                current_prefix = word  # 更新当前前缀
            elif flag == '$':  # 表示后缀
                if current_prefix is None:
                    continue  
                suffix_len = int(length)  # 后缀长度
                suffix = word[:suffix_len]  # 截取后缀
                candidate = current_prefix + suffix  # 拼接完整词
                if query_word == candidate:  # 判断是否匹配
                    found = True  # 找到了
                    break  # 跳出循环
            elif word:  # 单独的词
                candidate = word  # 直接使用
                if query_word == candidate:  # 判断是否匹配
                    found = True  # 找到了
                    break  # 跳出循环
        if found:  # 如果找到了
            break  # 跳出循环
        elif mid + 1 < len(index_positions) and query_word > block_str:  # 如果查询词大于当前块
            left = mid + 1  # 继续向右搜索
        else:  # 否则
            right = mid - 1  # 向左搜索

    return found  # 返回是否找到

def main():  # 主函数
    try:  # 异常捕获
        k = int(input("请输入块大小 k（1~87980）: "))  # 输入块大小
        if not (1 <= k <= 87980):  # 如果不在有效范围内
            print("错误：k 必须在 1 到 87980 之间")  # 输出错误信息
            return  # 退出程序

        query_word = input("请输入要查询的单词（仅限小写字母）: ").strip()  # 输入查询词
        if not is_valid_lowercase(query_word):  # 如果不符合格式
            print("错误：查询词必须为小写字母组成")  # 输出错误信息
            return  # 退出程序

        input_path = r"D:\code\natural_language_processing\lab5\compressed_dict.txt"  # 原始压缩字典路径
        output_path = fr"D:\code\natural_language_processing\lab5\compressed_dict_k={k}.txt"  # 新输出路径

        with open(input_path, 'r', encoding='utf-8') as f:  # 打开原始压缩文件
            compressed_str = f.read().strip()  # 读取并去除首尾空白

        full_compressed, index_positions, blocknumber = build_index(compressed_str, k)  # 构建新索引

        with open(output_path, 'w', encoding='utf-8') as f:  # 写入新的压缩文件
            f.write(full_compressed)  # 写入数据

        print(f"压缩后文件已保存至：{output_path}")  # 输出保存路径

        iterations = 1000  # 进行1000次测试
        start_time = time.perf_counter()  # 开始计时
        for _ in range(iterations):
            binary_search_query(full_compressed, index_positions, query_word)  # 查询测试
        end_time = time.perf_counter()  # 结束计时

        avg_time_ms = ((end_time - start_time) * 1000) / iterations  # 平均耗时计算（毫秒）
        print(f"平均查询时间为：{avg_time_ms:.4f} 毫秒")  # 输出平均时间

        new_length = len(full_compressed)  # 新压缩字符串长度
        overhead = (87980 + blocknumber) * 4  # 计算开销
        original_size = 2639400  # 原始词典大小

        compression_rate = (new_length + overhead) / original_size * 100  # 计算压缩率
        print(f"压缩率为：{compression_rate:.2f}%")  # 输出压缩率

    except Exception as e:  # 捕获异常
        print(f"发生错误：{e}")  # 输出错误信息

if __name__ == "__main__":  # 如果是主程序运行
    main()  # 调用主函数